In [32]:
import pandas as pd
import random
import numpy as np
import plotly
import phe.paillier as paillier
import time

In [ ]:
# Setting the random seed
np.random.seed(42)
rng = np.random.default_rng(42)
def randomizer(n, min_val, max_val):
    values = rng.integers(low = min_val, high = max_val, size = n)
    return values

main_list = randomizer(5, 0, 1000000)
print(main_list[:5])

[ 89250 773956 654571 438878 433015]


In [44]:
def baseline_mean(n, min_val, max_val):
    temp_list = randomizer(n, min_val, max_val).tolist()
    
    t0 = time.perf_counter()
    average = np.mean(temp_list)
    t1 = time.perf_counter()
    
    computation_time = t1 - t0

    return computation_time

n_list = [10, 20, 50, 100, 200, 500, 1000]
for i in n_list:
    computation_time = baseline_mean(i, 0, 1000000)
    print(f"n = {i:6d} | Total Runtime = {computation_time:.6f}s")

n =     10 | Total Runtime = 0.000035s
n =     20 | Total Runtime = 0.000017s
n =     50 | Total Runtime = 0.000023s
n =    100 | Total Runtime = 0.000013s
n =    200 | Total Runtime = 0.000016s
n =    500 | Total Runtime = 0.000025s
n =   1000 | Total Runtime = 0.000042s


In [33]:
public_key, private_key = paillier.generate_paillier_keypair()

In [58]:
def paillier_mean(n, min_val, max_val, public_key, private_key):
    temp_list = randomizer(n, min_val, max_val).tolist()

    t0 = time.perf_counter()
    encrypted_values = []
    for i in temp_list:
        encrypted_values.append(public_key.encrypt(i))
    encrypted_sum = sum(encrypted_values)
    decrypted_sum = private_key.decrypt(encrypted_sum)

    average = decrypted_sum / n
    t1 = time.perf_counter()
    computation_time = t1 - t0

    return computation_time

for i in n_list:
    computation_time = paillier_mean(i, 0, 1000000, public_key, private_key)
    print(f"n = {i:6d} | Total Runtime = {computation_time:.6f}s")

n =     10 | Total Runtime = 2.213501s
n =     20 | Total Runtime = 4.334214s
n =     50 | Total Runtime = 10.764304s
n =    100 | Total Runtime = 21.586259s
n =    200 | Total Runtime = 43.219625s
n =    500 | Total Runtime = 109.669847s
n =   1000 | Total Runtime = 220.466283s


In [ ]:
def shamir_share_generator(secret, party_count, threshold, min_val, max_val):
    coeffs = [secret] + rng.integers(low = min_val, high = max_val, size = threshold - 1).tolist()
    shares = []

    for x in range(1, party_count + 1):
        y = sum(coeff * (x ** power) for power, coeff in enumerate(coeffs))
        shares.append((x, y))  # store (party index, share value)
    return shares 

def shamir_reconstruct_secret(shares, threshold):
    secret = 0
    for j, (xj, yj) in enumerate(shares[:threshold]):
        lj = 1
        for m, (xm, _) in enumerate(shares[:threshold]):
            if m != j:
                lj *= xm / (xm - xj)
        secret += yj * lj

    return round(secret) 

shares = shamir_share_generator(12345, 5, 3, 1, 1000000)
print(shares)
secret_reconstructed = shamir_reconstruct_secret(shares, 3)
print("Reconstructed secret:", secret_reconstructed)

[(1, 425125), (2, 1531389), (3, 3331137), (4, 5824369), (5, 9011085)]
Reconstructed secret: 12345
